# Notebook 6 — Otimização de Carteiras e Backtest com Rebalanceamento

Construção e avaliação *out-of-sample* das estratégias de carteira, a partir dos retornos
saneados (NB4–5) e do estimador de covariância decidido no NB6. **Este notebook produz o
`strategy_returns.csv`**, insumo central da inferência estatística (NB11).

## Decisões metodológicas

**Estratégias.** Quatro carteiras comparáveis mais o benchmark de mercado:
- **EqualWeight (1/N)** — baseline; difícil de superar (DeMiguel, Garlappi & Uppal, 2009).
- **Mínima Variância (GMV)** — minimiza $w'\Sigma w$; dispensa $\mu$, robusta ao erro de estimação.
- **Máximo Sharpe (tangência)** — maximiza $(w'\mu - r_f)/\sqrt{w'\Sigma w}$; usa $\mu$ (ruidoso).
- **Inverso da Volatilidade** — $w_i \propto 1/\sigma_i$; baseline de paridade de risco ingênua.
- **IBOVESPA** — benchmark de mercado.

**Protocolo sem lookahead.** Janela de estimação móvel de **252 pregões**; **rebalanceamento
mensal**. Os pesos em $t$ usam exclusivamente dados até $t-1$ (último pregão do mês anterior),
e são aplicados aos retornos realizados do mês seguinte — garantindo avaliação estritamente
fora da amostra.

**Restrições.** *Long-only* ($w_i\ge 0$) e totalmente investida ($\sum_i w_i = 1$), coerente com
mandatos típicos e evitando alavancagem extrema da carteira de tangência.

**Covariância.** Estimada por **Ledoit-Wolf** em cada janela (decisão do NB6), garantindo
$\Sigma$ bem-condicionada e invertível mesmo com $N$ próximo de $T$.

**Custos de transação.** Custo proporcional ao *turnover* a cada rebalanceamento, descontado do
retorno, com os pesos sofrendo *drift* entre rebalanceamentos. A taxa é um parâmetro explícito.

## 1. Configuração e parâmetros

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize

parent_path = Path.cwd().parent.parent
DIR_RETORNOS = parent_path / "data" / "Retornos"
OUTPUT_DIR   = parent_path / "data" / "Estrategias"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Parâmetros do backtest (explícitos para auditoria) ---
TRADING_DAYS     = 252
JANELA_ESTIMACAO = 252       # pregões de lookback para μ e Σ
REBAL            = "M"       # frequência de rebalanceamento: "M" mensal, "Q" trimestral
METODO_COV       = "ledoit_wolf"
CUSTO_BPS        = 20.0      # custo por unidade de turnover (pontos-base); calibrável
LONG_ONLY        = True

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"[OK] Janela: {JANELA_ESTIMACAO}d | Rebalance: {REBAL} | Cov: {METODO_COV} | "
      f"Custo: {CUSTO_BPS} bps/turnover | Long-only: {LONG_ONLY}")

[OK] Janela: 252d | Rebalance: M | Cov: ledoit_wolf | Custo: 20.0 bps/turnover | Long-only: True


## 2. Carregamento dos insumos

In [2]:
def _ler(nome, col=None):
    pq, csv = DIR_RETORNOS / f"{nome}.parquet", DIR_RETORNOS / f"{nome}.csv"
    df = (pd.read_parquet(pq) if pq.exists()
          else pd.read_csv(csv, index_col=0, parse_dates=True) if csv.exists()
          else pd.read_csv(Path.cwd()/f"{nome}.csv", index_col=0, parse_dates=True))
    df.index = pd.to_datetime(df.index)
    return df[col] if col else df

ret  = _ler("retornos_simples_saneado").sort_index()
ibov = _ler("ibov_retornos", "ibov_ret_simples").sort_index()
rf   = _ler("rf_diario", "cdi_diario").sort_index()
N = ret.shape[1]
print(f"Retornos: {ret.shape[0]} pregões × {N} ativos ({ret.index.min().date()} → {ret.index.max().date()})")

Retornos: 3966 pregões × 131 ativos (2010-01-05 → 2025-12-30)


## 3. Estimador de covariância (Ledoit-Wolf vetorizado) e funções de pesos

In [10]:
def ledoit_wolf(X):
    """X: T×N. Encolhimento p/ identidade escalada (Ledoit & Wolf, 2004). Vetorizado."""
    X = np.asarray(X, float); T, n = X.shape
    Xc = X - X.mean(0)
    S = (Xc.T @ Xc) / T
    mu = np.trace(S) / n
    F = mu * np.eye(n)
    d2 = np.sum((S - F) ** 2) / n
    sq = (Xc ** 2).sum(1)
    quad = np.einsum('ti,ij,tj->t', Xc, S, Xc)
    b2bar = (np.sum(sq ** 2) - 2 * np.sum(quad) + T * np.sum(S ** 2)) / (n * T ** 2)
    delta = min(b2bar, d2) / d2 if d2 > 0 else 0.0
    return delta * F + (1 - delta) * S

def estimar_sigma(janela):
    if METODO_COV == "amostral":
        return np.cov(janela, rowvar=False)
    return ledoit_wolf(janela)

_bounds = [(0.0, 1.0)] * N if LONG_ONLY else [(-1.0, 1.0)] * N
_cons   = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)

def w_equal():       return np.repeat(1.0 / N, N)
def w_inv_vol(S):    iv = 1.0 / np.sqrt(np.diag(S)); return iv / iv.sum()
def w_min_var(S):
    r = minimize(lambda w: w @ S @ w, w_equal(), method="SLSQP",
                 bounds=_bounds, constraints=_cons, options={"maxiter": 200, "ftol": 1e-10})
    return r.x / r.x.sum()
def w_max_sharpe(mu, S, rf_a):
    def neg_sharpe(w):

        #v = np.sqrt(w @ S @ w)
        v = np.sqrt(TRADING_DAYS)
        
        return -((w @ mu - rf_a) / v) if v > 0 else 0.0
    r = minimize(neg_sharpe, w_equal(), method="SLSQP",
                 bounds=_bounds, constraints=_cons, options={"maxiter": 300, "ftol": 1e-10})
    return r.x / r.x.sum()

ESTRATEGIAS = ["EqualWeight", "MinVar", "MaxSharpe", "InvVol"]
print("Funções de pesos definidas:", ESTRATEGIAS)

Funções de pesos definidas: ['EqualWeight', 'MinVar', 'MaxSharpe', 'InvVol']


## 4. Backtest com rebalanceamento (sem lookahead, com custos e *drift*)

Em cada período: estima $\mu,\Sigma$ na janela que termina no último pregão do período anterior,
calcula os pesos-alvo, e aplica-os aos retornos realizados do período corrente. O *turnover* é
medido contra os pesos que sofreram *drift* desde o último rebalanceamento, e o custo é
descontado no primeiro pregão do período.

In [11]:
periodos = ret.index.to_period(REBAL)
uperiodos = pd.Index(periodos.unique())

ret_estrategias = {k: [] for k in ESTRATEGIAS}
pesos_hist = {k: {} for k in ESTRATEGIAS}
turnover_hist = {k: [] for k in ESTRATEGIAS}
w_ant = {k: None for k in ESTRATEGIAS}
custo_unit = CUSTO_BPS / 1e4

for i in range(1, len(uperiodos)):
    dias = ret.index[periodos == uperiodos[i]]
    fim_prev = ret.index[periodos == uperiodos[i - 1]][-1]
    janela = ret.loc[:fim_prev].iloc[-JANELA_ESTIMACAO:]
    if len(janela) < JANELA_ESTIMACAO:
        continue
    S  = estimar_sigma(janela.values)
    mu = janela.mean().values * TRADING_DAYS
    rf_a = rf.loc[janela.index].mean() * TRADING_DAYS
    alvos = {"EqualWeight": w_equal(), "MinVar": w_min_var(S),
             "MaxSharpe": w_max_sharpe(mu, S, rf_a), "InvVol": w_inv_vol(S)}
    R = ret.loc[dias].values                      # T_periodo × N
    for k, w in alvos.items():
        # turnover vs pesos com drift do período anterior
        if w_ant[k] is None:
            turn = 1.0
        else:
            turn = np.abs(w - w_ant[k]).sum()
        turnover_hist[k].append((dias[0], turn))
        # retornos do período com pesos fixos; custo no 1º dia
        rp = R @ w
        rp = rp.astype(float).copy()
        rp[0] -= custo_unit * turn
        ret_estrategias[k].append(pd.Series(rp, index=dias))
        pesos_hist[k][dias[0]] = w
        # drift dos pesos até o fim do período (p/ turnover do próximo rebalance)
        cresc = np.prod(1 + R, axis=0)
        w_drift = w * cresc
        w_ant[k] = w_drift / w_drift.sum()

strategy_returns = pd.DataFrame({k: pd.concat(v) for k, v in ret_estrategias.items()})
strategy_returns["IBOV"] = ibov.reindex(strategy_returns.index)
strategy_returns.index.name = "data"
print(f"Backtest concluído: {strategy_returns.shape[0]} pregões OOS, "
      f"{strategy_returns.index.min().date()} → {strategy_returns.index.max().date()}")
print(f"Rebalanceamentos efetivos: {len(turnover_hist['MinVar'])}")

Backtest concluído: 3700 pregões OOS, 2011-02-01 → 2025-12-30
Rebalanceamentos efetivos: 179


## 5. Métricas de desempenho (líquidas de custo, out-of-sample)

In [12]:
def metricas(r, rf_serie):
    r = r.dropna()
    rf_m = rf_serie.reindex(r.index).mean()
    #ann  = r.mean() * TRADING_DAYS

    # Calcule o CAGR a partir do retorno acumulado final:
    cum = (1 + r).cumprod()
    cagr = (cum.iloc[-1] ** (TRADING_DAYS / len(r))) - 1
    # Substitua no dicionário de retorno: "ret_anual": cagr

    vol  = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = (r.mean() - rf_m) / r.std() * np.sqrt(TRADING_DAYS)
    #downside = r[r < 0].std() * np.sqrt(TRADING_DAYS)
    downside = np.sqrt((r.clip(upper=0)**2).mean()) * np.sqrt(TRADING_DAYS)
    sortino = (r.mean() - rf_m) * TRADING_DAYS / downside if downside > 0 else np.nan
    cum = (1 + r).cumprod()
    maxdd = (cum / cum.cummax() - 1).min()
    return {"ret_anual": cagr, "vol_anual": vol, "sharpe": sharpe,
            "sortino": sortino, "max_drawdown": maxdd}

desempenho = pd.DataFrame({c: metricas(strategy_returns[c], rf) for c in strategy_returns.columns}).T
# turnover médio anualizado por estratégia
reb_por_ano = {"M": 12, "Q": 4}.get(REBAL, 12)
for k in ESTRATEGIAS:
    t = np.mean([v for _, v in turnover_hist[k][1:]])   # ignora o 1º (alocação inicial)
    desempenho.loc[k, "turnover_aa"] = t * reb_por_ano
desempenho.loc["IBOV", "turnover_aa"] = np.nan

print("Desempenho out-of-sample (líquido de custos):\n")
print(desempenho.round(4).to_string())

Desempenho out-of-sample (líquido de custos):

             ret_anual  vol_anual  sharpe  sortino  max_drawdown  turnover_aa
EqualWeight     0.0395     0.1864 -0.1969  -0.2727       -0.5310       0.8808
MinVar          0.0831     0.1135 -0.0577  -0.0809       -0.3434       2.9622
MaxSharpe       0.0082     0.5148  0.0923   0.1359       -0.8900       9.0337
InvVol          0.0740     0.1764 -0.0333  -0.0464       -0.4836       0.8579
IBOV            0.0620     0.2331 -0.0228  -0.0320       -0.4682          NaN


## 6. Exportação — `strategy_returns.csv` (insumo do NB11) e diagnósticos

In [13]:
def _salvar(df, nome):
    df.to_csv(OUTPUT_DIR / f"{nome}.csv", date_format="%Y-%m-%d", float_format="%.8f")
    try:
        df.to_parquet(OUTPUT_DIR / f"{nome}.parquet", engine="pyarrow")
        print(f"  {nome}.csv + .parquet")
    except Exception as e:
        print(f"  {nome}.csv  [parquet: {e}]")

print("[+] Exportando em:", OUTPUT_DIR, "\n")
_salvar(strategy_returns, "strategy_returns")          # << insumo do NB11
_salvar(desempenho, "desempenho_estrategias")

# Histórico de pesos (long format) e turnover
linhas = []
for k in ESTRATEGIAS:
    for data, w in pesos_hist[k].items():
        for ticker, peso in zip([c.replace("ACAO_", "") for c in ret.columns], w):
            if peso > 1e-6:
                linhas.append({"estrategia": k, "data": data, "ticker": ticker, "peso": peso})
pd.DataFrame(linhas).to_csv(OUTPUT_DIR / "pesos_historico.csv", index=False, float_format="%.6f")
print(f"  pesos_historico.csv ({len(linhas)} linhas)")

print("\n" + "="*60)
print("NOTEBOOK 7 CONCLUÍDO")
print("="*60)
print(f"strategy_returns: {strategy_returns.shape[0]} pregões × {strategy_returns.shape[1]} séries")
print("PRÓXIMO: plugar strategy_returns no NB11 (substitui o placeholder sintético).")
print("="*60)

[+] Exportando em: c:\VSCodeWorkspace\TCC_Final\data\Estrategias 

  strategy_returns.csv + .parquet
  desempenho_estrategias.csv + .parquet
  pesos_historico.csv (53610 linhas)

NOTEBOOK 7 CONCLUÍDO
strategy_returns: 3700 pregões × 5 séries
PRÓXIMO: plugar strategy_returns no NB11 (substitui o placeholder sintético).


## 7. Apêndice para o TCC — texto de redação

> *"As carteiras foram construídas e avaliadas estritamente fora da amostra, mediante janela de
> estimação móvel de 252 pregões e rebalanceamento mensal: os pesos de cada período utilizaram
> apenas informação disponível até o pregão anterior, sendo aplicados aos retornos realizados do
> período seguinte. Adotaram-se quatro estratégias — pesos iguais (1/N), mínima variância,
> máximo índice de Sharpe e inverso da volatilidade —, todas com restrição de não-negatividade e
> plena alocação, comparadas ao Ibovespa. A matriz de covariância de cada janela foi estimada
> pelo encolhimento de Ledoit e Wolf (2004). Custos de transação proporcionais ao *turnover*
> foram descontados a cada rebalanceamento, com os pesos sofrendo deriva entre as datas. As
> séries de retorno resultantes constituem a base da inferência estatística subsequente."*

**Artefatos em `data/Estrategias/`:** `strategy_returns.*` (insumo do NB11),
`desempenho_estrategias.*` e `pesos_historico.csv`.

**Observação (ligação com o NB6):** o vetor $\mu$ por média histórica é ruidoso, o que torna a
carteira de Máximo Sharpe sensível e tipicamente concentrada — daí a importância de testar, no
NB11, se seu desempenho superior é estatisticamente significativo ou fruto de sorte amostral.